# Real-world Data Wrangling
## Weather and bike-sharing demand in Washington, D.C.

This notebook gathers two public datasets, assesses two quality issues and two structural issues, validates the cleaning, and answers a daily ridership question with two visualizations. Raw inputs are preserved separately from cleaned results; the analysis is descriptive.

In [ ]:
from pathlib import Path
import calendar, hashlib, importlib.util, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from IPython.display import display
for folder in ['raw', 'cleaned', 'figures']:
    Path(folder).mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

In [ ]:
# Cached raw inputs let the submitted notebook run without fetching data again.
if not Path('raw/uci_bikeshare_hourly.csv').exists():
    if importlib.util.find_spec('ucimlrepo') is None:
        import subprocess
        dependency_dir = str(Path('project_dependencies').resolve())
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '--no-deps', '--target', dependency_dir, 'ucimlrepo==0.0.7'])
        sys.path.insert(0, dependency_dir)
    from ucimlrepo import fetch_ucirepo

**Reproducibility:** Run the cells in order from the project root. The UCI CSV is included; the NOAA station file is downloaded on first use; if the UCI input is absent, the official `ucimlrepo` client is used for API retrieval. Required libraries are pandas, NumPy, Matplotlib, seaborn and requests.

## 1. Gather data
Two sources are gathered using different methods: the UCI Python API client and a programmatic NOAA file download.

### 1.1 Problem statement
How do daily bike rental counts vary with temperature and measured precipitation in Washington, D.C., during 2011–2012? I combine UCI Capital Bikeshare hourly rental counts with daily observations from NOAA’s Washington Reagan Airport station. Comparisons use dates with all 24 coded hours and observed weather, with the airport station serving as a regional weather proxy. The results describe associations rather than causal weather effects.

**Sources and documentation:**
- [UCI Bike Sharing dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) and [official UCI Python client](https://github.com/uci-ml-repo/ucimlrepo).
- [NOAA station file](https://www.ncei.noaa.gov/pub/data/ghcn/daily/all/USW00013743.dly), [GHCN-Daily format and missing-value documentation](https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt), and [NOAA station identification](https://www.ncei.noaa.gov/access/services/data/v1?dataset=normals-monthly-1991-2020&stations=USW00013743&startDate=0001-01-01&endDate=9996-12-31&format=pdf).

### 1.2 Gathering methods
Both datasets exceed 500 source records and contain multiple variables. Cached inputs preserve the retrieved versions, while the gathering cells retain executable retrieval code.

### Dataset 1: Capital Bikeshare hourly counts
I selected the UCI Bike Sharing dataset because its dated hourly counts describe rental demand during 2011–2012. It was gathered through the official UCI Python API client and saved as an unmodified CSV representation containing 17,379 rows and 17 variables. `dteday` and `hr` identify the observation time, `cnt` is total rentals, `casual` and `registered` separate user counts, `workingday` identifies the calendar category, and `instant` identifies the source record.

In [ ]:
# Gathering method 1: retrieve UCI dataset 275 through its official Python API client.
if Path('raw/uci_bikeshare_hourly.csv').exists():
    bike_raw = pd.read_csv('raw/uci_bikeshare_hourly.csv')
else:
    bike_source = fetch_ucirepo(id=275)
    bike_raw = bike_source.data.original.copy()
    bike_raw.to_csv('raw/uci_bikeshare_hourly.csv', index=False)
    Path('raw/uci_bikeshare_metadata.json').write_text(json.dumps(bike_source.metadata, indent=2, default=str), encoding='utf-8')
print('Bike dataset:', bike_raw.shape)
display(bike_raw.head())

### Dataset 2: NOAA Washington Reagan Airport weather
I selected station USW00013743 because its Washington-area observations overlap the bike-sharing dates. Its fixed-width `.dly` file was downloaded with requests and parsed into 18,882 source records, each identified by station, year, month and weather element. The numbered daily value fields contain TMAX/TMIN in tenths of degrees Celsius and PRCP in tenths of millimeters; measurement, quality and source flags remain separate fields.

In [ ]:
# Gathering method 2: download NOAA's documented fixed-width station file.
NOAA_URL = 'https://www.ncei.noaa.gov/pub/data/ghcn/daily/all/USW00013743.dly'
weather_path = Path('raw/noaa_washington_reagan_USW00013743.dly')
if not weather_path.exists():
    response = requests.get(NOAA_URL, timeout=90)
    response.raise_for_status()
    weather_path.write_bytes(response.content)
lines = weather_path.read_text(encoding='ascii').splitlines()
assert all(len(line) == 269 for line in lines)
# Keep each source field separate; day-number headings expose the wide monthly structure.
records = []
for line in lines:
    record = {'station': line[:11], 'year': int(line[11:15]), 'month': int(line[15:17]), 'element': line[17:21]}
    for day in range(1, 32):
        offset = 21 + (day - 1) * 8
        record[f'value_{day:02d}'] = int(line[offset:offset + 5])
        record[f'mflag_{day:02d}'] = line[offset + 5]
        record[f'qflag_{day:02d}'] = line[offset + 6]
        record[f'sflag_{day:02d}'] = line[offset + 7]
    records.append(record)
weather_raw = pd.DataFrame(records)
print('Weather dataset:', weather_raw.shape, 'bytes:', weather_path.stat().st_size)
display(weather_raw.iloc[:5, :12])
assert len(bike_raw) > 500 and bike_raw.shape[1] >= 2
assert len(weather_raw) > 500 and weather_raw.shape[1] >= 2

### Raw-data storage
The original station file, bike CSV, UCI metadata and source manifest are stored under `raw/`. The manifest records retrieval information, gathering methods and SHA-256 hashes so subsequent cleaning can be checked against the preserved inputs.

In [ ]:
# Keep the original retrieval record when reusing stored raw inputs.
from datetime import datetime, timezone
manifest_path = Path('raw/source_manifest.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
else:
    manifest = {'retrieved_utc': datetime.now(timezone.utc).isoformat(),
                'sources': {'bike':'https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset', 'weather':NOAA_URL},
                'methods': {'bike':'UCI Python API client', 'weather':'programmatic fixed-width file download'},
                'files': {str(path): hashlib.sha256(path.read_bytes()).hexdigest()
                          for path in [Path('raw/uci_bikeshare_hourly.csv'), weather_path]}}
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Raw inputs saved:', list(manifest['files']))

## 2. Assess data
Each of the four issues below has a displayed source/example table and a separate programmatic check. Assessment views leave the raw inputs unchanged.

### Quality issue 1: Encoded missing weather measurements

In [ ]:
# Inspect original values before interpreting missing-value codes.
core = weather_raw.loc[weather_raw.element.isin(['TMAX', 'TMIN', 'PRCP'])]
display(core.loc[core.filter(regex=r'^value_').eq(-9999).any(axis=1), ['station','year','month','element','value_01','value_02','value_03']].head(8))

In [ ]:
# Build a read-only assessment view, keeping the source DataFrames unchanged.
def daily_view(monthly):
    view = pd.wide_to_long(monthly, stubnames=['value', 'mflag', 'qflag', 'sflag'],
                          i=['station','year','month','element'], j='day', sep='_', suffix=r'\d+').reset_index()
    view['date'] = pd.to_datetime(view[['year','month','day']], errors='coerce')
    return view.loc[view.date.notna()].copy()
weather_assessed = daily_view(core)
missing_mask = weather_assessed.value.eq(-9999)
print('Missing measurements on actual calendar days:', int(missing_mask.sum()))
print(weather_assessed.assign(missing=missing_mask).groupby('element').missing.agg(['sum','count']))
assert missing_mask.any()
print('Calendar padding excluded:', len(core) * 31 - len(weather_assessed))

**Issue and justification:** The source uses `-9999` for missing measurements; treating it as a real temperature or rainfall amount would distort summaries. Inspection of original monthly rows shows the marker, and the calendar-aware daily assessment counts 115 missing measurements across TMAX, TMIN and PRCP. Excluding 1,728 nonexistent-calendar-day slots prevents confusing format padding with missing observations; no data is fabricated.

### Quality issue 2: Incomplete coverage for comparable daily rental totals

In [ ]:
# Compare the documented 0–23 hour coding with observed date/hour pairs.
expected_dates = pd.date_range(pd.to_datetime(bike_raw.dteday).min(), pd.to_datetime(bike_raw.dteday).max(), freq='D')
expected_hours = pd.MultiIndex.from_product([expected_dates.strftime('%Y-%m-%d'), range(24)], names=['dteday','hr'])
observed_hours = pd.MultiIndex.from_frame(bike_raw[['dteday','hr']])
missing_hours = expected_hours.difference(observed_hours).to_frame(index=False)
display(missing_hours.head(12))
hour_coverage = bike_raw.groupby('dteday').hr.nunique().reindex(expected_dates.strftime('%Y-%m-%d'), fill_value=0)
display(hour_coverage.loc[hour_coverage < 24].head(10).rename('recorded_hours').to_frame())

In [ ]:
print('Absent date/hour slots:', len(missing_hours))
print('Days with all 24 coded hours:', int(hour_coverage.eq(24).sum()))
print('Days below the 24-slot benchmark:', int(hour_coverage.lt(24).sum()))
print('Repeated date/hour keys:', bike_raw.duplicated(['dteday','hr']).sum())
assert len(missing_hours) > 0
assert bike_raw.duplicated(['dteday','hr']).sum() == 0
assert bike_raw.hr.between(0,23).all()
# No observed count is replaced: omitted hours may reflect zero demand, closures,
# or timing/reporting conventions. The source does not identify the reason per gap.

**Issue and justification:** The 0–23 coding yields 165 absent date/hour slots across 76 dates; summing those dates alongside fully represented dates risks comparing different coverage. Displaying missing keys makes the gaps visible, while a date/hour grid and unique-key count quantify them. The reason for each omitted slot is not supplied: zero demand, closures and clock/reporting conventions are possible, so an absent record is not automatically zero.

### Tidiness issue 1: Daily observations spread across numbered columns

In [ ]:
# One source row holds up to 31 daily observations in numbered headings.
display(core[['station','year','month','element','value_01','value_02','value_03','value_04','value_05']].head())
display(weather_assessed[['station','date','element','value']].head())

In [ ]:
day_columns = core.filter(regex=r'^value_').columns
print('Numbered daily value columns:', len(day_columns))
print('Monthly source rows:', len(core), 'actual daily element observations:', len(weather_assessed))
assert len(day_columns) == 31
assert weather_assessed.groupby(['station','year','month','element']).size().max() > 1

**Issue and justification:** One NOAA station-month-element row contains up to 31 daily observations in `value_01` through `value_31`. The displayed wide table and count of 31 value columns confirm that daily observations are embedded in headings instead of forming rows. A calendar-aware long view shows the appropriate station-date-element observational unit while preserving the accompanying flags.

### Tidiness issue 2: Study-date attributes split across two sources

In [ ]:
# The same study dates have rental measurements in one source and weather in another.
display(bike_raw[['dteday','hr','cnt','casual','registered']].head())
display(weather_assessed.loc[weather_assessed.date.between('2011-01-01','2012-12-31'), ['station','date','element','value']].head(8))

In [ ]:
bike_dates = set(pd.to_datetime(bike_raw.dteday).dt.normalize())
weather_dates = set(weather_assessed.date)
print('Bike days:', len(bike_dates), 'matching weather dates:', len(bike_dates & weather_dates))
print('Date field types:', bike_raw.dteday.dtype, weather_assessed.date.dtype)
assert len(bike_dates & weather_dates) > 500
assert set(bike_raw.columns).isdisjoint({'TMAX','TMIN','PRCP'})
# Align daily observational units before merging to prevent hourly weather duplication.

**Issue and justification:** Rentals and weather for the same Washington study dates reside in separate datasets. Side-by-side source examples and 731 shared dates establish a meaningful join, but hourly rentals must first be aggregated to daily counts to match weather granularity. A one-to-one date join after this alignment prevents multiplication of daily counts.

## 3. Clean data
Cleaning uses copies and validates each operation immediately. A complete-hour cohort is a transparent treatment of incomplete coverage, not a claim that the underlying source records are wrong.

In [ ]:
bike_clean = bike_raw.copy()
weather_clean = weather_raw.loc[weather_raw.element.isin(['TMAX','TMIN','PRCP'])].copy()

### Quality issue 1: Decode missing weather values

In [ ]:
# Decode missing values without replacing them with invented weather.
value_columns = weather_clean.filter(regex=r'^value_').columns
weather_clean[value_columns] = weather_clean[value_columns].replace(-9999, np.nan)

In [ ]:
assert not weather_clean[value_columns].eq(-9999).any().any()
assert int(weather_clean[value_columns].isna().sum().sum()) == int(missing_mask.sum()) + (len(core) * 31 - len(weather_assessed))
print('Encoded missing values are now NaN; calendar padding will be removed during reshaping.')

**Justification:** Replace `-9999` with NaN, retaining true zero precipitation. The assertion checks that no numeric sentinel remains and reconciles the missing-cell count with the assessed missing observations plus calendar padding. Missing weather is not mean-imputed; the analysis later requires complete weather for retained dates.

### Quality issue 2: Restrict comparisons to complete-hour dates

In [ ]:
# Define a complete-hour cohort; absence is not assumed to mean zero demand.
complete_dates = hour_coverage.loc[hour_coverage.eq(24)].index
bike_clean = bike_clean.loc[bike_clean.dteday.isin(complete_dates)].copy()
bike_clean['date'] = pd.to_datetime(bike_clean['dteday'], format='%Y-%m-%d')
print('Retained dates:', bike_clean.date.nunique(), 'excluded dates:', len(hour_coverage) - len(complete_dates))

In [ ]:
assert bike_clean.groupby('date').hr.nunique().eq(24).all()
assert not bike_clean.duplicated(['date','hr']).any()
assert bike_clean.cnt.eq(bike_clean.casual + bike_clean.registered).all()
assert len(bike_clean) == len(complete_dates) * 24
print('Daily comparisons have complete coded-hour coverage; no count was imputed.')

**Justification:** Retain the 655 dates with all 24 distinct coded hours, excluding 76 dates rather than guessing absent counts. Unique date/hour keys, 24-hour coverage and the identity `cnt = casual + registered` are asserted. This is a conservative comparison cohort; it also excludes legitimate shorter clock days and may select against low-demand or interrupted dates.

### Tidiness issue 1: Reshape daily weather observations

In [ ]:
# Reshape month/day-number columns to one station-date-element observation per row.
weather_clean = daily_view(weather_clean)
weather_clean['value'] = weather_clean['value'] / 10.0
weather_clean['unit'] = np.where(weather_clean.element.eq('PRCP'), 'mm', 'degrees C')
display(weather_clean[['station','date','element','value','unit','mflag','qflag','sflag']].head())

In [ ]:
assert not weather_clean.duplicated(['station','date','element']).any()
assert weather_clean.date.notna().all()
assert weather_clean['value'].dropna().ne(-9999).all()
assert weather_clean.loc[weather_clean.element.eq('PRCP'), 'value'].dropna().ge(0).all()
assert len(weather_clean) == len(weather_assessed)
print('Daily observation keys are unique and values have documented physical units.')

**Justification:** Convert the numbered daily fields to rows, construct actual dates, and discard nonexistent calendar-day padding. Values are divided by ten into documented physical units; flags remain available for provenance. Unique station-date-element keys, valid dates, nonnegative observed rainfall and reconciled row counts validate the result.

### Tidiness issue 2: Combine aligned daily observations

In [ ]:
# Align daily units before joining: sum rentals, pivot distinct weather variables.
assert bike_clean.groupby('date').workingday.nunique().eq(1).all()
bike_daily = bike_clean.groupby('date', as_index=False).agg(
    rentals=('cnt','sum'), casual=('casual','sum'), registered=('registered','sum'),
    hours_observed=('hr','nunique'), working_day=('workingday','first'))
weather_daily = weather_clean.pivot(index='date', columns='element', values='value').rename(
    columns={'TMAX':'tmax_c','TMIN':'tmin_c','PRCP':'precip_mm'}).reset_index()
weather_daily.columns.name = None
combined = bike_daily.merge(weather_daily, on='date', how='left', validate='one_to_one', indicator=True)
print(combined['_merge'].value_counts())
weather_complete = combined[['tmax_c','tmin_c','precip_mm']].notna().all(axis=1)
print('Complete-hour dates with missing weather excluded:', int((~weather_complete).sum()))
combined = combined.loc[weather_complete].drop(columns='_merge').copy()

In [ ]:
assert combined.date.is_unique
assert combined.hours_observed.eq(24).all()
assert combined[['tmax_c','tmin_c','precip_mm']].notna().all().all()
assert combined.tmax_c.ge(combined.tmin_c).all()
assert combined.rentals.eq(combined.casual + combined.registered).all()
assert len(combined) > 500 and combined.shape[1] >= 4
print('Validated daily merged dataset:', combined.shape)

**Justification:** Sum hourly rentals only within the complete-hour cohort and pivot the distinct weather variables into daily columns. A left date merge with `validate="one_to_one"` finds all 655 weather matches; zero cohort dates lack required weather. Checks verify unique dates, correct coverage, matching user-count totals and TMAX not below TMIN.

### Retain variables needed for the question
The final dataset keeps date, year, working-day category, rental counts, coverage and weather variables. Redundant source IDs, repeated month descriptors, normalized UCI weather variables and nonanalysis station elements are removed from the merged table, while the raw inputs and cleaned weather flags remain stored. The temperature midpoint `(TMAX + TMIN) / 2` is an explicitly derived summary, not a separately observed daily mean.

In [ ]:
# Retain variables supporting the question and remove redundant source descriptors.
combined['year'] = combined.date.dt.year
combined['temperature_midpoint_c'] = (combined.tmax_c + combined.tmin_c) / 2
final_data = combined[['date','year','working_day','rentals','casual','registered',
                       'hours_observed','tmax_c','tmin_c','temperature_midpoint_c','precip_mm']].sort_values('date').reset_index(drop=True)
assert final_data.isna().sum().sum() == 0
display(final_data.head())

## 4. Update the data store
`raw/` preserves source versions; `cleaned/` contains the complete-hour daily rentals, long weather observations and final merged dataset. Informative filenames and a numeric/date schema support reuse, and a CSV round-trip check validates saved values and types. Source hashes must remain unchanged.

In [ ]:
# Keep both source inputs and cleaned products with descriptive names.
bike_daily.to_csv('cleaned/bikeshare_complete_hour_daily.csv', index=False, encoding='utf-8')
weather_clean[['station','date','element','value','unit','mflag','qflag','sflag']].to_csv(
    'cleaned/noaa_weather_daily_long.csv', index=False, encoding='utf-8')
final_data.to_csv('cleaned/washington_bikeshare_weather_2011_2012.csv', index=False, encoding='utf-8')
csv_dtypes = {column:str(dtype) for column,dtype in final_data.dtypes.items() if column != 'date'}
Path('cleaned/final_dataset_schema.json').write_text(json.dumps({'date':'datetime64[ns]', **csv_dtypes}, indent=2), encoding='utf-8')
reloaded = pd.read_csv('cleaned/washington_bikeshare_weather_2011_2012.csv', parse_dates=['date'], dtype=csv_dtypes)
pd.testing.assert_frame_equal(final_data, reloaded, check_exact=False, rtol=1e-10)
assert all(hashlib.sha256(Path(path).read_bytes()).hexdigest() == digest for path,digest in manifest['files'].items())
print('Saved final dataset:', final_data.shape, 'raw hashes unchanged; CSV round trip passed.')

## 5. Answer the research question
The two figures use the final cleaned daily dataset. Years are separated to reduce conflation with system growth; these exploratory comparisons do not control for every seasonal or behavioral factor.

**Research question:** How do daily bike rental counts vary with temperature and measured precipitation in Washington, D.C., during 2011–2012, on dates with all 24 coded hours?

In [ ]:
# Separate years because growth in the system can confound an overall association.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
for ax, year in zip(axes, [2011, 2012]):
    subset = final_data.loc[final_data.year.eq(year)]
    sns.regplot(data=subset, x='temperature_midpoint_c', y='rentals', ax=ax, ci=None,
                scatter_kws={'alpha':0.5, 's':22}, line_kws={'color':'firebrick'})
    correlation = subset.temperature_midpoint_c.corr(subset.rentals)
    ax.set(title=f'{year}: n={len(subset)}, Pearson r={correlation:.2f}',
           xlabel='Daily temperature midpoint (degrees C)', ylabel='Rentals per day')
fig.suptitle('Washington bike rentals and temperature on complete-hour dates')
fig.tight_layout()
fig.savefig('figures/temperature_ridership_by_year.png', bbox_inches='tight')
plt.show()
print(final_data.groupby('year').apply(lambda x: pd.Series({'days':len(x),
      'temperature_rentals_correlation':x.temperature_midpoint_c.corr(x.rentals)})))

**Visual 1 — Temperature:** Within the complete-hour cohort, daily rentals are positively associated with the temperature midpoint in both 2011 (305 days; Pearson r = 0.715) and 2012 (350 days; r = 0.713), as the upward point patterns and fitted lines show. The separate panels also reveal higher rental levels in 2012, so these associations should be interpreted alongside growth, seasonality and the cohort restriction rather than as a causal effect of temperature.

In [ ]:
# Compare measured precipitation groups within each year, not causal effects.
plot_data = final_data.assign(precipitation_group=np.where(final_data.precip_mm.gt(0), '>0 mm', '0 mm reported'))
fig, ax = plt.subplots(figsize=(8, 4.7))
sns.boxplot(data=plot_data, x='precipitation_group', y='rentals', hue='year',
            order=['0 mm reported', '>0 mm'], ax=ax)
ax.set(title='Daily rentals by measured precipitation and year',
       xlabel='Daily precipitation (mm)', ylabel='Rentals per day')
ax.legend(title='Year')
fig.tight_layout()
fig.savefig('figures/precipitation_ridership_by_year.png', bbox_inches='tight')
plt.show()
rain_summary = plot_data.groupby(['year','precipitation_group']).rentals.agg(['count','median','mean'])
print(rain_summary.round(2))

**Visual 2 — Precipitation:** Median rentals on days with measured precipitation above zero are 3,559 versus 4,304 for the zero-reported group in 2011, and 5,142.5 versus 6,458.5 in 2012—approximately 17.3% and 20.4% lower, respectively. The within-year boxplots support a negative descriptive association with precipitation, but the groups overlap, zero-reported days can include traces, and seasonal or working-day differences may also contribute.

### 5.2 Reflection

With more time, I would investigate omitted hourly records with the data provider and explicit timezone information to distinguish zero demand, closures and daylight-saving conventions. I would assess the sensitivity of the results to incomplete-hour dates and compare multiple weather stations across the service area. I would also model season, working days and system growth alongside weather to check whether the descriptive associations persist. These steps would improve coverage and interpretation, but observational data would still require care before making causal claims.